In [ ]:
# This script uses class definition to store, access and compute GP Observables in an organized manner

import GPKoopman as gpk
import torch
import numpy as np
import matplotlib.pyplot as plt
import math
import datetime
import time

if not torch.cuda.is_available():
    print('Cuda not available')

### Cost Function Definition

In [ ]:
# Multi-Trajectory Cost Function Definition

def get_cost_AC(Z, X, Xplus, Xtrain, manager, nT=1, lambda1=1.0, lambda2=1.0, lambda3=1.0):
    """
    Computes the cost function using a single differentiable GP forward pass per observable,
    merging the training and prediction steps by passing Z[:, i] directly to the forward method.
    
    Args:
        Z: Tensor of shape (r**n, p), decision variable (requires grad).
        X: Tensor of shape (n, nT*N), dataset of N steps per trajectory.
        Xplus: Tensor of shape (n, nT*N), time-shifted dataset.
        Xtrain: Tensor of shape (n, r**n), gridpoints for training.
        manager: GPObservablesManager.
        nT: Number of trajectories.
        lambda1: Weighting for NLPD.
        lambda2: Weighting for linearity enforcement.
        lambda3: Weighting for prediction error minimization.
    """
    N = X.shape[1] // nT    # Number of time steps per trajectory
    p = Z.shape[1]          # Number of observables
    l = Z.shape[0] // nT    # Decision horizon
    n = X.shape[0]          # State dimension

    # For each observable, call forward once on the full dataset X (and Xplus)
    M = torch.empty((p, N * nT), device=X.device)
    cov_all = [None] * p  # store full covariance matrices for X
    Mplus = torch.empty((p, N * nT), device=X.device)
    # cov_all_plus = [None] * p  # store full covariance matrices for Xplus

    for i in range(p):
        mean_i, cov_i = manager.observables[i].forward(X, Z[:, i])
        M[i, :] = torch.transpose(mean_i, 0, -1)
        cov_all[i] = cov_i

        mean_plus_i, _ = manager.observables[i].forward(Xplus, Z[:, i])
        Mplus[i, :] = torch.transpose(mean_plus_i, 0, -1)

    # Compute the pseudo-inverse lifting operator and the corresponding matrices Cz and Az.
    M_pinv = torch.linalg.pinv(M)
    Cz = X @ M_pinv
    Az = Mplus @ M_pinv

    # Cost Term 1: Negative Log Predictive Density (NLPD)
    NormNLPD = 0.0
    if not math.isclose(lambda1, 0):
        for j in range(nT):
            TrajNLPD = 0.0
            # Define the number of time steps for NLPD computation:
            num_steps = N - 2 - l
            vz_k = torch.empty((p, num_steps), device=X.device)
            for i in range(p):
                # For trajectory j, determine the indices for the NLPD slice.
                start = j * N + l + 1
                end = (j + 1) * N - 1  # end is exclusive
                # Extract the covariance for the slice from the precomputed full covariance.
                cov_sub = cov_all[i][start:end, start:end]
                # Get the diagonal elements (predictive variances) and clamp them.
                vz_k[i, :] = torch.clamp(torch.diag(cov_sub), min=1e-3)
            for k in range(num_steps):
                vx_next = torch.abs(torch.diag(Cz @ Az @ torch.diag(vz_k[:, k]) @ Az.T @ Cz.T).view(n, 1))
                error_term = ((X[:, j * N + l + 1 + k + 1] - Cz @ Az @ M[:, j * N + l + 1 + k]) ** 2) / vx_next
                log_term = torch.log(vx_next)
                TrajNLPD += torch.sum(error_term + log_term)
            NormNLPD += TrajNLPD

    # Cost Term 2: Linearity Enforcement
    NormLEP = 0.0
    if not math.isclose(lambda2, 0):
        for j in range(nT):
            TrajLEP = 0.0
            for k in range(l - 1):
                lin_error = torch.transpose(Z[j * l + k + 1, :], 0, -1) - Az @ torch.transpose(Z[j * l + k, :], 0, -1)
                TrajLEP += torch.norm(lin_error)
            NormLEP += TrajLEP

    # Cost Term 3: Prediction Error Minimization
    NormPEM = 0.0
    if not math.isclose(lambda3, 0):
        for j in range(nT):
            TrajPEM = 0.0
            for k in range(N - 1):
                pred_error = X[:, j * N + (k + 1)] - Cz @ (torch.linalg.matrix_power(Az, k + 1)) @ M[:, j * N]
                TrajPEM += torch.norm(pred_error)
            NormPEM += TrajPEM

    cost = (lambda1 * NormNLPD / ((N - l) * nT)) + (lambda2 * NormLEP / (l * nT)) + (lambda3 * NormPEM / (N * nT))
    return cost

def get_cost_varsep(Z, X, Xplus, manager, nT=1):
    """
    Calculate the 2-norm between next-step lifted state and linearly propagated lifted state
    Args:
        Z:
        X:
        Xplus:
        Xtrain:
        manager:
        nT:
    """
    N = X.shape[1] // nT    # Number of time steps per trajectory
    p = Z.shape[1]          # Number of observables
    l = Z.shape[0] // nT    # Decision horizon
    n = X.shape[0]          # State dimension

    # For each observable, call forward once on the full dataset X (and Xplus)
    M = torch.empty((p, N * nT), device=X.device)
    Mplus = torch.empty((p, N * nT), device=X.device)
    
    for i in range(p):
        mean_i, _ = manager.observables[i].forward(X, Z[:, i])
        M[i, :] = torch.transpose(mean_i, 0, -1)

        mean_plus_i, _ = manager.observables[i].forward(Xplus, Z[:, i])
        Mplus[i, :] = torch.transpose(mean_plus_i, 0, -1)
    
    # 2-norm
    mmt = M @ M.T
    try:
        L = torch.linalg.cholesky(mmt)
        invMMt = torch.cholesky_inverse(L)
    except RuntimeError:
        invMMt = torch.linalg.inv(mmt)

    Mpinv = M.T @ invMMt
    Mplus_bar = torch.vstack([Mplus, X])
    err = Mplus_bar - (Mplus_bar @ Mpinv @ M)
    err_norm = torch.norm(err)
    return err_norm


def cost_prop_theta(X, Xplus, manager, nz=1, nT=1, Ngp=None):
    """
    Calcuate the single-step propagation based cost in the lifted space for optimal Kernel-HP computation
    """
    # n_s = nT * Ngp
    N = X.shape[1] / nT
    if Ngp is None:
        Ngp = int(0.1 * N)
    nT, N, Ngp, nz = int(nT), int(N), int(Ngp), int(nz)
    device = manager.observables[0].device
    G = torch.empty((nT * N, nT * Ngp * nz), device=device)
    Gplus = torch.empty((nT * N, nT * Ngp * nz), device=device)
    for i in range(nz):
        G[:, i*Ngp*nT:(i+1)*nT*Ngp] = manager.observables[i].forward_G(X)
        Gplus[:, i*Ngp*nT:(i+1)*nT*Ngp] = manager.observables[i].forward_G(Xplus)
    # Gt = torch.transpose(G, 0, 1)
    gram = G.T @ G
    try:
        L = torch.linalg.cholesky(gram)
        invGtG = torch.cholesky_inverse(L)
    except RuntimeError:
        invGtG = torch.linalg.inv(gram)

    eye_N = torch.eye(N * nT, device=device)

    Gcost1 = torch.linalg.matrix_norm(eye_N - G @ invGtG @ G.T)
    Gcost2 = torch.linalg.matrix_norm(Gplus)
    return math.sqrt(nz) * Gcost1 * Gcost2 / (N * nT * Ngp)
    # G_err = (eye_N - G @ invGtG @ Gt) @ Gplus
    # Gcost = math.sqrt(nz) * torch.linalg.matrix_norm(G_err)
    # return Gcost


def cost_prop_theta2(X, Xplus, manager, nz=1, nT=1):
    """
    Compute the propagation-based GP cost per trajectory and return the average.
    """
    # total timepoints = nT * N  ⇒  N = X.shape[1] // nT
    nT, nz = int(nT), int(nz)
    N = X.shape[1] // nT
    device = manager.observables[0].device

    traj_costs = []
    I_N = torch.eye(N, device=device)
    for j in range(nT):
        # slice out the j-th trajectory
        start, end = j * N, (j + 1) * N
        Xj      = X    [:, start:end]
        Xp_j    = Xplus[:, start:end]

        # for each observable, get its feature block
        G_blocks, Gp_blocks = [], []
        for obs in manager.observables.values():
            G_blocks .append(obs.forward_G(Xj))      # shape (N, n_i)
            Gp_blocks.append(obs.forward_G(Xp_j))    # shape (N, n_i)

        # concatenate horizontally
        Gj      = torch.cat(G_blocks,  dim=1)  # (N, Σ n_i)
        Gplus_j = torch.cat(Gp_blocks, dim=1)  # (N, Σ n_i)

        # build Gram and its inverse (Cholesky if possible)
        gram = Gj.T @ Gj
        try:
            L = torch.linalg.cholesky(gram)
            inv_gram = torch.cholesky_inverse(L)
        except RuntimeError:
            inv_gram = torch.linalg.inv(gram)

        # cost terms for this trajectory
        cost1j = torch.linalg.matrix_norm(I_N - Gj @ inv_gram @ Gj.T)
        cost2j = torch.linalg.matrix_norm(Gplus_j)

        traj_costs.append(cost1j * cost2j)

    return math.sqrt(nz) * torch.stack(traj_costs).mean()


def cost_prop_theta_fast(X, Xplus, manager, nz=1, nT=1):
    """
    Fully vectorized across trajectories (no Python loops over j),
    returns sqrt(nz) * mean_j [ ||I - P_j|| * ||Gplus_j|| ].
    """
    nT, nz = int(nT), int(nz)
    N     = X.shape[1] // nT
    device = manager.observables[0].device

    # 1) Precompute G_full and Gplus_full for each observable on the *entire* data
    G_full_blocks  = []
    Gp_full_blocks = []
    for obs in manager.observables.values():
        # obs.forward_G returns shape (nT*N, n_i)
        G_full  = obs.forward_G( X    )   # (nT*N, n_i)
        Gp_full = obs.forward_G( Xplus)   # (nT*N, n_i)

        # reshape into a batch of per-trajectory feature maps (nT, N, n_i)
        G_full_blocks .append(G_full .view(nT, N, -1))
        Gp_full_blocks.append(Gp_full.view(nT, N, -1))

    # 2) Concatenate features from *all* observables into (nT, N, F)
    G_batched  = torch.cat(G_full_blocks,  dim=2)  # (nT, N, F)
    Gp_batched = torch.cat(Gp_full_blocks, dim=2)  # (nT, N, F)

    # 3) Build the batch of Gram matrices: (nT, F, F)
    #    and invert each one via Cholesky (or fallback to full inverse).
    Gram = G_batched.transpose(1, 2) @ G_batched    # (nT, F, F)
    try:
        # batched Cholesky & inverse
        L       = torch.linalg.cholesky(Gram)               # (nT, F, F)
        invGram = torch.cholesky_inverse(L)                 # (nT, F, F)
    except RuntimeError:
        invGram = torch.linalg.inv(Gram)                    # (nT, F, F)

    # 4) Compute the batch of projection matrices P_j = G_j invGram_j G_j^T → (nT, N, N)
    P = G_batched @ invGram @ G_batched.transpose(1, 2)    # (nT, N, N)

    # 5) Compute per-trajectory norms:
    I_N = torch.eye(N, device=device).unsqueeze(0)         # (1, N, N)
    # Frobenius norm over last two dims for each batch element
    cost1 = torch.linalg.vector_norm(I_N - P,    dim=(1,2))  # (nT,)
    cost2 = torch.linalg.vector_norm(Gp_batched, dim=(1,2))  # (nT,)

    # 6) average over trajectories
    return math.sqrt(nz) * (cost1 * cost2).mean()


def cost_recon_Z(Zcal, manager, X, nT=1, Ngp=None):
    """
    Calculate the reconstruction based cost for virtual target selection
    """
    nz = Zcal.shape[0]  # Zcal = \mathcal{Z}, constructed as block-diagonal of different Z.T
    N = X.shape[1] / nT
    if Ngp is None:
        Ngp = int(0.1 * N)
    nT, N, Ngp, nz = int(nT), int(N), int(Ngp), int(nz)
    G = torch.empty((nT * N, nT * Ngp * nz), device=manager.observables[0].device)
    for i in range(nz):
        G[:, i*Ngp*nT:(i+1)*nT*Ngp] = manager.observables[i].forward_G(X)
    Phi = Zcal @ torch.transpose(G,0,1)
    PhiT = torch.transpose(Phi,0,1)
    try:
        L = torch.linalg.cholesky(Phi @ PhiT)
        invPPt = torch.cholesky_inverse(L)
    except RuntimeError:
        invPPt = torch.linalg.inv(Phi @ PhiT)

    eye_N = torch.eye(N * nT, device=manager.observables[0].device)
    Zerr = X @ (eye_N - PhiT @ invPPt @ Phi)
    Zcost = torch.linalg.matrix_norm(Zerr)
    return Zcost

# Execution Cells

## Data Loading

In [ ]:
# Allowed system names -
# "Unforced Duffing" | "Unforced Duffing_right" | "van der Pol" | "Simple Pendulum"
# "Lorenz" | "Lotka Volterra" | "Piecewise Linear"

system_name = 'Simple Pendulum'
data = torch.load(f"Data/DataAuto_{system_name}.pt", weights_only=True)

SimData = data["trajectories"] # Shape: (num_trajectories, state_dim, num_steps)
ts = data["sample_time"]
num_trajectories = data["num_trajectories"]
N = data["num_steps"]

nTrain, nTest = math.floor(num_trajectories * 0.4), math.floor(num_trajectories * 0.2)

# Clip Training Data steps
# SimData = SimData[:,:,:151]
# N = 150

## Optimization Section

### Optimization Setup

In [ ]:
SimData = SimData.float()
SimData = SimData.to(device='cuda:0')
# Original State Dim | Lifted State Dim | Learning Horizon | Resolution
n, p, l, r = SimData.shape[1], 8, 10, 10

Xall = torch.cat([SimData[j, :, :] for j in range(nTrain)], dim=1)      # Concatenated total matrix
X = torch.cat([SimData[j, :, 0:N] for j in range(nTrain)], dim=1)       # Concatenated Data matrix
Xplus = torch.cat([SimData[j, :, 1:] for j in range(nTrain)], dim=1)    # Time-shifted Data matrix

ICsetTrain = torch.cat([SimData[j, :, 0].view(n,1) for j in range(nTrain)], dim=1)    # Random IC set for training
ICsetTest = torch.cat([SimData[j, :, 0].view(n,1) for j in range(nTrain, nTrain + nTest)], dim=1)  # Random IC set for testing

# Options: 'Horizon' | 'Grid' | 'K-Means'
trainMethod = 'Horizon'

# Initialize GP training-grid and decision variables
if trainMethod == 'Horizon':
    sample_idx = torch.linspace(0, N-1, steps=l, device=X.device).round().long()
    Xtrain = torch.cat([X[:, j*N + sample_idx] for j in range(nTrain)], dim=1)
    # Xtrain = torch.cat([X[:,j*N:j*N+l] for j in range(nTrain)],dim=1)
    #Z = torch.rand(Xtrain.shape[1], p, requires_grad=True)
    Z = torch.nn.Parameter(torch.rand(Xtrain.shape[1], p, device='cuda:0'))
    ObsManager = gpk.GPObservablesManager()
    for i in range(p):
        ObsManager.add_observable(index=i, d=n, ns=l*nTrain, kernel_types=['Gaussian'], combination='sum',noise=1e-4, m=500)
    for i in range(p):
        ObsManager.train_observable(i, Xtrain, Z[:, i])
    ObsManager.set_random_hyperparameters(scale=[1., 5., None])
    print('Observable Hyperparameters have been randomized:')
    ObsManager.print_parameters()
    Zcal = torch.block_diag(*[Z[:,i].T for i in range(p)])

elif trainMethod == 'K-Means':
    Xtrain = torch.cat([X[:,j*N:j*N+l] for j in range(nTrain)],dim=1)
    Z = torch.nn.Parameter(torch.rand(Xtrain.shape[1], p, device='cuda:0'))
    ObsManager = gpk.GPObservablesManager()
    centroids = gpk.get_kmeans(X, num_centers=p)
    for i in range(p):
        ObsManager.add_observable(index=i, d=n, ns=l*nTrain, kernel_types=['ExplicitAttractor', 'Gaussian'], combination='product',noise=1e-4, m=500)
    
    mu_centroids = [centroids[:, i:i+1] for i in range(centroids.shape[1])]
    mu_centroids.extend(mu_centroids)
    ObsManager.set_parameters(mu_list=mu_centroids)
    for i in range(p):
        ObsManager.train_observable(i, Xtrain, Z[:, i])
    # ObsManager.set_random_hyperparameters(scale=[1., 0.1, None])
    hp1_val, hp2_val = 1.0, 0.5
    hp1_val = [torch.tensor([hp1_val], device='cuda:0') for _ in range(2*p)]
    hp2_val = [torch.tensor([hp2_val], device='cuda:0') for _ in range(p)]
    hp2_val.extend([torch.tensor([0.001], device='cuda:0') for _ in range(p)])
    ObsManager.set_parameters(hp1_list=hp1_val, hp2_list=hp2_val)
    # print('Observable Hyperparameters have been randomized:')
    ObsManager.print_parameters()

    for i in range(p):
        plt.plot(centroids[0,i].cpu(), centroids[1,i].cpu(), marker='o')
    plt.grid()
    plt.title('Centroids of K-Means Clusters'), plt.xlabel('X1'), plt.ylabel('X2')

    Zcal = torch.block_diag(*[Z[:,i].T for i in range(p)])

else:
    raise ValueError(f'Unrecognized GP Training method {trainMethod}')


#### Optional Plotting

In [ ]:
# Plot the Xtrain points in 2D space
if Xtrain.shape[0] >= 2:    # Optional Plotting in 2D space
    Xtrain_plot = Xtrain.cpu()
    plt.figure(figsize=(4,4))
    for i in range(Xtrain.shape[1]):
        plt.plot(Xtrain_plot[0,i], Xtrain_plot[1,i], linestyle='None', marker='x', color='red', alpha=0.5)
    # plt.plot(X[0,:N].cpu(), X[1,:N].cpu())
    plt.xlabel('X1'), plt.ylabel('X2'), plt.grid()
    plt.title('Inducing Points (Xtrain)')
    plt.show()

### Verify Cost Function Works

In [ ]:
manualVerify = None
if manualVerify:
    manager = ObsManager
    nT, Ngp, nz = nTrain, l, p

    device = manager.observables[0].device

    # 1) Precompute G_full and Gplus_full for each observable on the *entire* data
    G_full_blocks  = []
    Gp_full_blocks = []
    for obs in manager.observables.values():
        # obs.forward_G returns shape (nT*N, n_i)
        G_full  = obs.forward_G( X    )   # (nT*N, n_i)
        Gp_full = obs.forward_G( Xplus)   # (nT*N, n_i)

        # reshape into a batch of per-trajectory feature maps (nT, N, n_i)
        G_full_blocks .append(G_full .view(nT, N, -1))
        Gp_full_blocks.append(Gp_full.view(nT, N, -1))

    # 2) Concatenate features from *all* observables into (nT, N, F)
    G_batched  = torch.cat(G_full_blocks,  dim=2)  # (nT, N, F)
    Gp_batched = torch.cat(Gp_full_blocks, dim=2)  # (nT, N, F)

    # 3) Build the batch of Gram matrices: (nT, F, F)
    #    and invert each one via Cholesky (or fallback to full inverse).
    Gram = G_batched.transpose(1, 2) @ G_batched    # (nT, F, F)
    invGram = torch.linalg.inv(Gram)
    P = G_batched @ invGram @ G_batched.transpose(1, 2)    # (nT, N, N)

    # 5) Compute per-trajectory norms:
    I_N = torch.eye(N, device=device).unsqueeze(0)         # (1, N, N)
    # Frobenius norm over last two dims for each batch element
    cost1 = torch.linalg.vector_norm(I_N - P,    dim=(1,2))  # (nT,)
    cost2 = torch.linalg.vector_norm(Gp_batched, dim=(1,2))  # (nT,)
    # G = torch.empty((nT * N, nT * Ngp * nz), device=manager.observables[0].device)
    # Gplus = torch.empty((nT * N, nT * Ngp * nz), device=manager.observables[0].device)
    # for i in range(nz):
    #     G[:, i*Ngp*nT:(i+1)*nT*Ngp] = manager.observables[i].forward_G(X)
    #     Gplus[:, i*Ngp*nT:(i+1)*nT*Ngp] = manager.observables[i].forward_G(Xplus)
    # Gt = torch.transpose(G, 0, 1)
    # G0 = manager.observables[0].forward_G(X)
    # Gnz = manager.observables[nz-1].forward_G(X)
    # try:
    #     L = torch.linalg.cholesky(Gt @ G)
    #     invGtG = torch.cholesky_inverse(L)
    # except RuntimeError:
    #     invGtG = torch.linalg.inv(Gt @ G)
    
    # print(f'Largest singular value condition number of G0 is: {torch.linalg.cond(G0, p=2)}')
    # print(f'Smallest singluar value condition number of G0 is: {torch.linalg.cond(G0, p=-2)}')
    # print(f'Frobenius condition number of GtG is: {torch.linalg.cond(Gt @ G, p='fro')}')
    # frob_norm = torch.linalg.matrix_norm(Gplus, ord='fro')
    # print(f'Frobenius norm of Gplus: {frob_norm}')
    # eye_N = torch.eye(N * nT, device=manager.observables[0].device)
    # print(f'Frobenius norm of first part: {torch.linalg.matrix_norm(eye_N - G @ invGtG @ Gt, ord='fro')}')

    # gpk.utilities.MatViz(G0, plot_type='heat')
    # gpk.utilities.MatViz(G0, plot_type='surf')


In [ ]:
verifyCost = 1
if verifyCost:
    t_start = time.perf_counter()
    cost1 = cost_prop_theta_fast(X, Xplus, ObsManager, nz=p, nT=nTrain)
    t_cost1 = time.perf_counter() - t_start

    print(f'Cost-1 [Propagation] is {cost1.cpu().detach()}, calculated in {t_cost1}s')

    t_start = time.perf_counter()
    cost2 = cost_recon_Z(Zcal, ObsManager, X, nT=nTrain, Ngp=l)
    t_cost2 = time.perf_counter() - t_start
    print(f'Cost-1 [Reconstruction] is {cost2.cpu().detach()}, calucated in {t_cost2}s')

## Kernel-HP Optimization

### Main Optimization Loop

In [ ]:
# Optimization Parameters: Maximum Iterations | Learning Rate | Target Error
max_iter, learn_rate, err_thresh = 500, 0.01, 0.1

lambda1, lambda2, lambda3 = 1., 10., 0.
# No-Improvement stopping criterion: Iters to monitor | Min decrease in cost
patience, min_delta = 10, 5e-1

print('Starting Iteration Loop!')
cost_history, iter, count_insignificant = [], 0, 0

# Gather parameters from each observable in the manager.
all_hp1, all_hp2, all_noise = [], [], []
for obs in ObsManager.observables.values():
    all_hp1.extend(list(obs.hp1_list))
    all_hp2.extend(list(obs.hp2_list))
    all_noise.append(obs.noise)

optimizer = torch.optim.Adam(all_hp1 + all_hp2, lr=learn_rate)
# optimizer = torch.optim.SGD(all_hp1 + all_hp2, lr=learn_rate, momentum=0.9, nesterov=True)
while iter < max_iter:
    # Reset gradients to zero
    optimizer.zero_grad()
    
    # cost = get_cost_AC(Z, X, Xplus, Xtrain, ObsManager, nT=nTrain, lambda1=lambda1, lambda2=lambda2, lambda3=lambda3)   # compute cost
    cost = cost_prop_theta(X, Xplus, ObsManager, nz=p, nT=nTrain, Ngp=l)
    cost_history.append(cost.item())    # add to cost history
    cost.backward(retain_graph=False)    # backpropagate
    for i, theta in enumerate(all_hp1): # clip hp1 gradients
        if theta.grad is not None:
            theta.grad.data = torch.nan_to_num(theta.grad.data)
            theta.grad.data = torch.clamp(theta.grad.data, min=-1e2, max=1e2)
            # print(f"hp1 parameter {i} grad: {theta.grad}")
            
        else:
            print(f"hp1 parameter {i} has no gradient.")
    
    for i, theta in enumerate(all_hp2): # clip hp2 gradients
        if theta.grad is not None:
            theta.grad.data = torch.nan_to_num(theta.grad.data)
            theta.grad.data = torch.clamp(theta.grad.data, min=-1e2, max=1e2)
            # print(f"hp2 parameter {i} grad: {theta.grad}")

        else:
            print(f"hp2 parameter {i} has no gradient.")

    optimizer.step()    # gradient descent step

    print(f"Iteration {iter}/{max_iter}")
    print(f"Cost: {cost.item()}")

    # ObsManager.print_parameters()
    iter += 1

if iter == max_iter:    # Print Stopping Criterion
    print(f'Stopping: Reached maximum number of iterations = {iter}.')

print('Optimization Complete.')
print("Final Cost:", cost.item())

### View Optimization Logs

In [ ]:
# Plot cost history
plt.figure(figsize=(6,4))
plt.plot(cost_history, label="Cost")
plt.title("Cost History")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.legend(), plt.grid()
plt.show()

# Logarithmic cost history
plt.figure(figsize=(6,4))
plt.plot(torch.log10(torch.tensor(cost_history)), label="log(Cost)")
plt.title("Logarithmic Cost History")
plt.xlabel("Iteration")
plt.ylabel("log(Cost)")
plt.legend(), plt.grid()
plt.show()

In [ ]:
ObsManager.print_parameters()

## Virtual Target Optimization

In [ ]:
# Optimize Virtual Targets using the optimal Kernel Hyperparameters
max_iter2, learn_rate2 = 500, 0.04
cost_history2, iter2 = [], 0

print('Starting Iteration Loop!')
t_start = time.perf_counter()
# optimizer2 = torch.optim.Adam([Z], lr=learn_rate2)
optimizer2 = torch.optim.SGD([Z], lr=learn_rate2, momentum=0.9, nesterov=True)
while iter2 < max_iter2:
    Zcal = torch.block_diag(*[Z[:,i].T for i in range(p)])
    optimizer2.zero_grad()  # Clear gradients
    cost2 = cost_recon_Z(Zcal, ObsManager, X, nT=nTrain, Ngp=l)  # compute cost
    cost_history2.append(cost2.item())    # add to cost history
    cost2.backward(retain_graph=False)    # backpropagate
    optimizer2.step()    # gradient descent step
    print(f"Iteration {iter2 + 1}/{max_iter2}")
    print(f"Cost: {cost2.item()}")

    # Increment iteration
    iter2 += 1
t_end = time.perf_counter()
optimal_Z = Z.detach()
print(f'Virtual Target optimization completed in {(t_end - t_start):.1f} seconds')
print(f'Final Cost: {cost2.item()}')

In [ ]:
# Plot cost history
plt.figure(figsize=(6,4))
plt.plot(cost_history2, label="Cost")
plt.title("Cost History")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.legend(), plt.grid()
plt.show()

# Logarithmic cost history
plt.figure(figsize=(6,4))
plt.plot(torch.log10(torch.tensor(cost_history2)), label="log(Cost)")
plt.title("Logarithmic Cost History")
plt.xlabel("Iteration")
plt.ylabel("log(Cost)")
plt.legend(), plt.grid()
plt.show()

In [ ]:
# extract optimal_Z from optimal_Zcal


## Post-Process Optimization Results

In [ ]:
# Use Optimal Z values to Build GP Models and Optimal A and Z matrices
optimal_Z = Z.detach()
for i in range(p):
    ObsManager.train_observable(i, Xtrain, optimal_Z[:,i])  # train GP Observables with Optimal Z outputs

# ObsManager.optimize_hyperparameters(opt_mu=False, opt_sigma=True, max_iter=100)   # Optimize Kernel hyperparameters for Optimal training data
# print(f'GPO Hyperparameters have been optimized.')
ObsManager.print_parameters()

ObsList = [i for i in range(p)]
A, C = gpk.getKoopman(ObsManager, ObsList, Xall, nTrain, stateAug=False)

# Simulation and Validation

## Model Simulation

In [ ]:
A, C = A.to(device='cpu'), C.to(device='cpu')
SimData = SimData.to(device='cpu')
ICsetTrain, ICsetTest = ICsetTrain.to(device='cpu'), ICsetTest.to(device='cpu')
# Evaluation on training set
ZmeanTrain = torch.empty((nTrain, p, N))    # n+p for state-augmentation
ZcvTrain = torch.empty((nTrain, p, p, N))   # n for no state-augmentation
#ZmeanTrain[:, :n, 0] = ICsetTrain.T    # only for state-augmentation

XhatTrain = torch.empty((nTrain, n, N))
XcvhatTrain = torch.empty((nTrain, n, n, N))
TrainRMSE = torch.empty((nTrain,n))

for j in range(nTrain): # GP Predict for all training trajectories
    for i in range(p):  # GP predict IC and IC-cv
        ZmeanTrain[j, i, 0] = ObsManager.predict_mean(i, ICsetTrain[:, j].view(n,1))        # n+i for state-augmentation
        ZcvTrain[j, i, i, 0] = ObsManager.predict_covariance(i, ICsetTrain[:, j].view(n,1)) # n for no state-augmentation
    
    ZmeanTrain[j, :, :], ZcvTrain[j, :, :, :], XhatTrain[j, :, :], XcvhatTrain[j, :, :, :] = gpk.sim_LTI(ZmeanTrain[j,:,0], A, C, num_steps=N, ts=None, x0cv=ZcvTrain[j,:,:,0])
    TrainRMSE[j,:] = torch.sqrt(torch.mean((XhatTrain[j,:,:] - SimData[j,:,:N])**2,1))

# Evaluation on test set
ZmeanTest = torch.empty((nTest, p, N))
ZcvTest = torch.empty((nTest, p, p, N))
#ZmeanTest[:, :n, 0] = ICsetTest.T  # only for state-augmentation

XhatTest = torch.empty((nTest, n, N))
XcvhatTest = torch.empty((nTest, n, n, N))
TestRMSE = torch.empty((nTest,n))

for j in range(nTest): # GP Predict for all testing trajectories
    for i in range(p):  # GP predict IC and IC-cv
        ZmeanTest[j, i, 0] = ObsManager.predict_mean(i, ICsetTest[:, j].view(n,1))          # n+i for state-augmentation
        ZcvTest[j, i, i, 0] = ObsManager.predict_covariance(i, ICsetTest[:, j].view(n,1))   # n for no state-augmentation

    ZmeanTest[j, :, :], ZcvTest[j, :, :, :], XhatTest[j, :, :], XcvhatTest[j, :, :, :] = gpk.sim_LTI(ZmeanTest[j,:,0], A, C, num_steps=N, ts=None, x0cv=ZcvTest[j,:,:,0])
    TestRMSE[j,:] = torch.sqrt(torch.mean((XhatTest[j,:,:] - SimData[nTest+j,:,:N])**2,1))

XhatTrain, XhatTest, XcvhatTrain, XcvhatTest = XhatTrain.detach(), XhatTest.detach(), XcvhatTrain.detach(), XcvhatTest.detach()
TestRMSE, TrainRMSE = TestRMSE.detach(), TrainRMSE.detach()

In [ ]:
# Function to compute RMSE, NLPD, and sRMSE


time = torch.arange(0., ts * N, ts)
idx1, idx2, idx3 = torch.argmin(TrainRMSE.mean(dim=1)), torch.argmin(TestRMSE.mean(dim=1)), torch.argmax(TestRMSE.mean(dim=1))

## Trajectory and Time-Evolution Plots

In [ ]:
gpk.plot_phase(XhatTrain, SimData, ICsetTrain, idx1, N, system_name, 'Training Trajectory')
gpk.plot_time_series_with_bounds(time, XhatTrain, XcvhatTrain, SimData, idx1, N, system_name, title_suffix='Best Train Trajectory')

gpk.plot_phase(XhatTest, SimData, ICsetTest, idx2, N, system_name, 'Best Test Trajectory', sim_offset=nTrain)
gpk.plot_time_series_with_bounds(time, XhatTest, XcvhatTest, SimData, idx2, N, system_name, title_suffix='Best Test Trajectory', sim_offset=nTrain)

gpk.plot_phase(XhatTest, SimData, ICsetTest, idx3, N, system_name, 'Worst Test Trajectory', sim_offset=nTrain)
gpk.plot_time_series_with_bounds(time, XhatTest, XcvhatTest, SimData, idx3, N, system_name, title_suffix='Worst Test Trajectory', sim_offset=nTrain)

In [ ]:
# For best test trajectory (e.g., idx2)
gpk.plot_predicted_sd_error(XcvhatTest, SimData, XhatTest, idx=idx2, N=N, nTrain=nTrain, trajectory_label="Best Test")

# For worst test trajectory (e.g., idx3)
gpk.plot_predicted_sd_error(XcvhatTest, SimData, XhatTest, idx=idx3, N=N, nTrain=nTrain, trajectory_label="Worst Test")

## System-level Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Training set plot
axes[0].plot(range(nTrain), TrainRMSE[:,0].numpy(), marker='o', linestyle='-', label='RMSE X1')
axes[0].plot(range(nTrain), TrainRMSE[:,1].numpy(), marker='o', linestyle='-', label='RMSE X2')
axes[0].set_title('Training Metrics')
axes[0].set_xlabel("Trajectory Index")
axes[0].set_ylabel("Metric Value")
axes[0].legend()
axes[0].grid()

# Test set plot
axes[1].plot(range(nTest), TestRMSE[:,0].numpy(), marker='o', linestyle='-', label='RMSE X1')
axes[1].plot(range(nTest), TestRMSE[:,1].numpy(), marker='o', linestyle='-', label='RMSE X2')
axes[1].set_title('Test Metrics')
axes[1].set_xlabel("Trajectory Index")
axes[1].set_ylabel("Metric Value")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# Eigen value plot of Koopman Matrices
eigval = torch.linalg.eigvals(A)

eigreal, eigimag = eigval.real, eigval.imag
eigreal, eigimag = eigreal.detach().numpy(), eigimag.detach().numpy()
eig_mag = np.sqrt(eigreal**2 + eigimag**2)

theta = np.linspace(0, 2*np.pi, 500)
unitCirclex, unitCircley = np.cos(theta), np.sin(theta)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
# First subplot: Eigenvalues plot
axes[0].plot(unitCirclex, unitCircley, color='orange', label='Unit Circle')
for i in range(np.size(eig_mag)):
    if eig_mag[i] <= 1:
        axes[0].scatter(eigreal, eigimag, color='green', label='Eigenvalues')
    else:
        axes[0].scatter(eigreal, eigimag, color='red', label='Eigenvalues')

axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_title(f"Eigenvalues of A Matrix with {p} Observables")
axes[0].set_xlabel("Real Part")
axes[0].set_ylabel("Imaginary Part")
axes[0].grid(True)
axes[0].legend(labels=['Unit Circle', 'Eignevalues'], loc='upper right')

# Second subplot: Heatmap of matrix A
im = axes[1].imshow(A.detach().numpy(), cmap='viridis', aspect='auto')
fig.colorbar(im, ax=axes[1], label="Value")
axes[1].set_title(f'{A.shape[0]}-D Koopman Matrix')
axes[1].set_xlabel("Columns")
axes[1].set_ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:
# Phase Diagram from all IC simulation
if SimData.shape[1] == 3:       # 3D plot
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    for j in range(SimData.shape[0]):
        ax.plot(SimData[j, 0, :], SimData[j, 1, :], SimData[j, 2, :],
                alpha=0.5, color='blue')
        ax.scatter(SimData[j, 0, 0], SimData[j, 1, 0], SimData[j, 2, 0],
                   color='red')
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')
    ax.set_zlabel('X3')
    ax.set_title(f'All Trajectories: {system_name}')
    ax.grid(True)
    plt.show()

elif SimData.shape[1] == 2:     # 2D plot
    for j in range(SimData.shape[0]):
        plt.plot(SimData[j, 0, :], SimData[j, 1, :],
                 alpha=0.5, color='blue', linewidth=1.)
        plt.plot(SimData[j, 0, 0], SimData[j, 1, 0],
                 'o', color='red')
    plt.grid()
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(f'All Trajectories: {system_name}')
    plt.show()

elif SimData.shape[1] == 1:     # 1D plot
    for j in range(SimData.shape[0]):
        plt.plot(SimData[j, 0, :], alpha=0.5, color='blue', linewidth=1.)
        plt.plot(SimData[j, 0, 0], 'o', color='red')
    plt.grid()
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(f'All Trajectories: {system_name}')
    plt.show()

else:   # system dimension > 3
    print('IC Phase Plots supported for 1/2/3-D systems only.')


# Training Set Predicted Trajectories
if XhatTrain.shape[1] == 3: # 3D plot
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    for j in range(XhatTrain.shape[0]):
        ax.plot(XhatTrain[j, 0, :], XhatTrain[j, 1, :], XhatTrain[j, 2, :],
                alpha=0.5, color='blue')
        ax.scatter(XhatTrain[j, 0, 0], XhatTrain[j, 1, 0], XhatTrain[j, 2, 0],
                   color='red')
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')
    ax.set_zlabel('X3')
    ax.set_title(f'Predicted Trajectories - Training Set: {system_name}')
    ax.grid(True)
    plt.show()

elif XhatTrain.shape[1] == 2:   # 2D plot
    for j in range(XhatTrain.shape[0]):
        plt.plot(XhatTrain[j, 0, :], XhatTrain[j, 1, :],
                 alpha=0.5, color='blue', linewidth=1.)
        plt.plot(XhatTrain[j, 0, 0], XhatTrain[j, 1, 0],
                 'o', color='red')
    plt.grid()
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(f'Predicted Trajectories - Training Set: {system_name}')
    plt.show()

else:
    print('Trianing Response Phase PLots supported for 2/3-D systems only.')

# Model Saving

In [ ]:
today = datetime.date.today()

iGPKResults = {
    "Sysem Name": system_name,
    "Data": SimData,
    "Train Test Factors": [nTrain, nTest],
    "Initial Conditions": [ICsetTrain, ICsetTest],
    "ObsManager": ObsManager,
    "Observables": ObsManager.observables,
    "Koopman A": A,
    "Koopman C": C,
    "X Train": Xtrain,
    "Optimal Z": optimal_Z,
    "Train RMSE": TrainRMSE,
    "Test RMSE": TestRMSE,
    "Adam Options": [learn_rate, max_iter, lambda1, lambda2, lambda3]
}

# torch.save(iGPKResults, f'Results/iGPKAuto_{system_name}_Noisy_{today}.pt')